In [13]:
import math
import os
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path

r_resolution = 1200 / 325 #On calcule la résolution verticale pixel/mm donc 1200pixel/325mm 

d_distance = 900 #Fixé dans le TD mais normalement environ 3x taille verticale donc ici 3x325

alpha_radians = math.radians(2) #Imite la fovéa de l'oeil, fixer entre 1.5 et 2

sigma = r_resolution * d_distance * math.tan(alpha_radians) #formule du TP : Gaussian Spread

def generate_partial_saliency_map(width, height, x_fix, y_fix, sigma_val):
    x_coords = np.arange(0, width, 1) #tous les pixels de la largeur danns un tableau de dim1
    y_coords = np.arange(0, height, 1) #tous les pixels de la hateur dans un tableau de dim1
    
    x_grid, y_grid = np.meshgrid(x_coords, y_coords) #tableau à partir des pixels de hauteur et largeur

    #Formule du TP
    x_term = ((x_grid - x_fix) ** 2) / (2 * (sigma_val ** 2))
    y_term = ((y_grid - y_fix) ** 2) / (2 * (sigma_val ** 2))
    
    # Calcul la partial silency map avec A = 1 (donc pas besoin de le mettre ici)
    partial_map = np.exp(-(x_term + y_term))
    
    return partial_map

def generate_global_saliency_map(width, height, fixations, sigma_val):
    global_map = np.zeros((height, width)) #On crée matrice vide de la taille de l'image
    
    #boucle sur les coords de fixations pour somme de m à Mfix
    for x_fix, y_fix in fixations:
        #Ajoute les valeurs de regard sur la matrice globale
        global_map += generate_partial_saliency_map(width, height, x_fix, y_fix, sigma_val)
        
    #On prends la valeur max pour la normalisation
    max_val = np.max(global_map)
    
    #Normalisation par la valeur max pour avoir qqchose etre 0 et 1
    if max_val > 0:
        global_map = global_map / max_val
        
    return global_map

def save_saliency_maps(image_path, fixations, sigma_val, output_dir):
    #On récupére ses dimensions
    orig_img = Image.open(image_path).convert("RGB")
    width, height = orig_img.size
    
    #Generates the mathematical saliency array for this image.
    saliency_map = generate_global_saliency_map(width, height, fixations, sigma_val)
    
    #On passe de valeur normalisée (0 à 1) à des valeurs standards couleur (0 à 255)
    grayscale_array = (saliency_map * 255).astype(np.uint8)
    #On convertit l'image en niveau de gris
    gray_img = Image.fromarray(grayscale_array, mode="L")
    
    color_map = plt.get_cmap("jet") #Valeur proche de 0 en bleu et proche de 1 en rouge
    colored_array = (color_map(saliency_map) * 255).astype(np.uint8)
    heatmap_img = Image.fromarray(colored_array, mode="RGBA")
    
    #canal alpha qui correspond à la transparence
    orig_rgba = orig_img.convert("RGBA")
    
    #Superposition de l'image de base avec l'image heatmap
    blended_img = Image.blend(orig_rgba, heatmap_img, alpha=0.5)
    
    #On récupère le nom de l'image pour la sauvegarde
    base_name = Path(image_path).name
    
    #On sauvegarde l'image en nivea ude gris
    gray_img.save(Path(output_dir) / f"gray_{base_name}")
    
    #On sauvegarde la heatmap
    blended_img.convert("RGB").save(Path(output_dir) / f"color_{base_name}")

#A changer si au CREMI.
images_dir = "MexCulture142/images_train" 
fixations_dir = "MexCulture142/fixations"
output_directory = "saliency_outputs"

#Pour save en local
os.makedirs(output_directory, exist_ok=True)

#On fait toutes les images du train dataset
for image_file in Path(images_dir).glob("*.png"):
    #On check le .txt correspondant
    corrected_stem = image_file.stem.replace("_N_", "_GazeFix_N_") #Ce n'est pas juste _N_ Mais _GazeFix_N_
    fixation_path = Path(fixations_dir) / f"{corrected_stem}.txt"
    
    # Affiche exactement où Python cherche et quel nom de fichier il attend
    print(f"Vérification : {fixation_path}") 
    
    if fixation_path.exists():
        current_fixations = []
        
        #On lit les coords depuis le fichier
        with open(fixation_path, "r") as file:
            for line in file:
                # Assuming the format contains comma-separated X and Y coordinates.
                parts = line.strip().split()
                if len(parts) >= 2:
                    x, y = float(parts[0]), float(parts[1])
                    current_fixations.append((x, y))
                    
        if current_fixations:
            save_saliency_maps(str(image_file), current_fixations, sigma, output_directory)
            print(f"Successfully processed maps for {image_file.name}")

Vérification : MexCulture142\fixations\Colonial_AcademiaDeBellasArtes_Queretaro_GazeFix_N_2.txt


C:\Users\MOI\AppData\Local\Temp\ipykernel_8848\2236705041.py:59: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  gray_img = Image.fromarray(grayscale_array, mode="L")
C:\Users\MOI\AppData\Local\Temp\ipykernel_8848\2236705041.py:63: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  heatmap_img = Image.fromarray(colored_array, mode="RGBA")


Successfully processed maps for Colonial_AcademiaDeBellasArtes_Queretaro_N_2.png
Vérification : MexCulture142\fixations\Colonial_AdvocacionesDeLaIglesiaDeSanSimon_Michoacan_GazeFix_N_1.txt
Successfully processed maps for Colonial_AdvocacionesDeLaIglesiaDeSanSimon_Michoacan_N_1.png
Vérification : MexCulture142\fixations\Colonial_AdvocacionesDeLaIglesiaDeSanSimon_Michoacan_GazeFix_N_2.txt
Successfully processed maps for Colonial_AdvocacionesDeLaIglesiaDeSanSimon_Michoacan_N_2.png
Vérification : MexCulture142\fixations\Colonial_AntiguaPresidenciaMunicipal_Chihuahua_GazeFix_N_2.txt
Successfully processed maps for Colonial_AntiguaPresidenciaMunicipal_Chihuahua_N_2.png
Vérification : MexCulture142\fixations\Colonial_AntiguoConventoJesuita_Michoacan_GazeFix_N_1.txt
Successfully processed maps for Colonial_AntiguoConventoJesuita_Michoacan_N_1.png
Vérification : MexCulture142\fixations\Colonial_AntiguoConventoJesuita_Michoacan_GazeFix_N_2.txt
Successfully processed maps for Colonial_AntiguoConv